# Solution B3 · Cost-complexity pruning chosen in cross-validation

**Type**: Implement with scikit-learn

## Objective

Use `cost_complexity_pruning_path` and `ccp_alpha` to prune a regression tree on the house prices, choose `alpha` in cross-validation (with the pruning **inside** each fold) applying both the minimum rule and the "one standard deviation" rule (the smallest tree within
one standard error of the best), and compare the number of leaves and the error.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.tree import DecisionTreeRegressor

X, y = ds.california()
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=0)
cv = KFold(5, shuffle=True, random_state=0)
base = DecisionTreeRegressor(min_samples_leaf=5, random_state=0)
alphas = np.geomspace(1e-5, 1e-2, 30)

## Commented solution

In [ ]:
tree = base.fit(X_tr, y_tr)
n_alphas = len(tree.cost_complexity_pruning_path(X_tr, y_tr).ccp_alphas)
initial_leaves = tree.get_n_leaves()
print(f"candidate alpha values: {n_alphas} | leaves of the unpruned tree: {initial_leaves}")

In [ ]:
cv_rmse, cv_se, leaves = {}, {}, {}
for a in alphas:
    scores = -cross_val_score(DecisionTreeRegressor(min_samples_leaf=5, random_state=0, ccp_alpha=a), X_tr, y_tr, cv=cv, scoring="neg_root_mean_squared_error")
    cv_rmse[a], cv_se[a] = scores.mean(), scores.std(ddof=1) / np.sqrt(5)
    leaves[a] = DecisionTreeRegressor(min_samples_leaf=5, random_state=0, ccp_alpha=a).fit(X_tr, y_tr).get_n_leaves()
cv_rmse, cv_se, leaves = pd.Series(cv_rmse), pd.Series(cv_se), pd.Series(leaves)
print(pd.DataFrame({"CV RMSE": cv_rmse, "standard error": cv_se, "leaves": leaves}).round(4))

In [ ]:
alpha_min = cv_rmse.idxmin()
alpha_1se = max(a for a in alphas if cv_rmse[a] <= cv_rmse.min() + cv_se[alpha_min])
print(f"alpha of the minimum: {alpha_min:.2e} | alpha with the 1-SE rule: {alpha_1se:.2e}")

In [ ]:
def try_alpha(a):
    m = DecisionTreeRegressor(min_samples_leaf=5, random_state=0, ccp_alpha=a).fit(X_tr, y_tr)
    return mean_squared_error(y_te, m.predict(X_te), squared=False), m.get_n_leaves()


(rmse_min, leaves_min), (rmse_1se, leaves_1se) = try_alpha(alpha_min), try_alpha(alpha_1se)
print(f"minimum: test RMSE {rmse_min:.3f}, {leaves_min} leaves | 1-SE: test RMSE {rmse_1se:.3f}, {leaves_1se} leaves")

## Check

In [ ]:
with verify():
    check(n_alphas is not None and n_alphas > 1500 and initial_leaves > 2000, "the base tree has over 2000 leaves and over 1500 candidate alpha values", "cost_complexity_pruning_path and get_n_leaves")
    check(len(cv_rmse) == 30 and len(cv_se) == 30 and len(leaves) == 30 and leaves.is_monotonic_decreasing and leaves.iloc[0] > 1000 and leaves.iloc[-1] < 30, "30 alpha values; the leaves decrease (from over 1000 to under 30)", "more alpha, more pruning")
    check(1e-4 <= alpha_min <= 1e-3 and alpha_1se >= alpha_min and cv_rmse[alpha_1se] <= cv_rmse.min() + cv_se[alpha_min] + 1e-12, "best alpha between 1e-4 and 1e-3; 1-SE alpha >= alpha_min and within one standard error", "the largest alpha within cv_rmse.min() + cv_se[alpha_min]")
    check(leaves_1se <= leaves_min and leaves_min < initial_leaves / 5, "the 1-SE tree has at most the leaves of the minimum one; both well below the initial leaves", "get_n_leaves()")
    check(rmse_1se < rmse_min + 0.03 and rmse_min < 0.66, "the 1-SE tree loses less than 0.03 of RMSE compared with the minimum (RMSE below 0.66)", "compare rmse_min and rmse_1se")
done()

## What you learned

- **`ccp_alpha` is a single continuous parameter for the complexity:** the full tree has over 2000 leaves and about 2000 candidate `alpha` values (each value corresponds to a subtree); as `alpha` grows the leaves drop from thousands to a handful. Pruning is a "softer" way of controlling the size compared with `max_depth`.
- **The pruning goes inside the cross-validation** (`ccp_alpha` is a parameter of the estimator, so each fold prunes independently): choosing `alpha` on the training set and then validating on the same folds is selection optimism.
- **The 1-SE rule picks the simplest model that cannot be told apart from the best:** the standard error (about 0.006) lets you move from the `alpha` of the minimum (4.5e-4, 199 leaves, test RMSE 0.639) to a slightly larger one (5.7e-4, 163 leaves, 0.645): 18% fewer leaves for 6 thousandths of RMSE. The rule does not always change the choice: with a sharp minimum the difference is small, with flat curves it can be very large. Fewer leaves = more readable, faster and more stable.
- **`ccp_alpha` depends on the scale of the target** (it is expressed in impurity units): the best value here is not transferable to a problem with the price in dollars instead of hundreds of thousands (exercise E3).

## Common pitfall

Choosing `ccp_alpha` by looking at the test set or at the same folds on which the base tree was chosen. Pruning is a hyperparameter like the others and requires a validation that has not already seen it.

## Interview question

> *Click on a question to show its answer.*

<details><summary>What is cost-complexity pruning?</summary>

I build the full tree and then minimize error + alpha·(number of leaves): for each alpha there is an optimal subtree and the subtrees are nested, so you get a sequence of smaller and smaller trees. I choose alpha in cross-validation, usually with the standard error rule
to prefer the simplest tree. Unlike stopping the growth early, it looks at the overall effect of a branch.

</details>